<a href="https://colab.research.google.com/github/Gulamali86/Freight-Rate-Prediction/blob/main/Freight_Rate_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install "matplotlib>=3.8,<4" "numpy>=1.26,<3" "pandas>=2.0,<3" lightgbm xgboost scikit-learn

In [5]:
import pandas as pd
import numpy as np

# 1. Load the training data
train_df = pd.read_csv("/content/train-test.csv")

# 2. Inspect the data structure
print("--- Training Data Overview ---")
print(train_df.info())
print("\n--- First 5 Rows ---")
print(train_df.head())
print("\n--- Missing Values ---")
print(train_df.isnull().sum())
print("\n--- Target Variable Summary ---")
print(train_df.describe())

--- Training Data Overview ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   load_id       48000 non-null  object 
 1   pickup        48000 non-null  object 
 2   delivery      48000 non-null  object 
 3   pickup_lat    48000 non-null  float64
 4   pickup_lon    48000 non-null  float64
 5   delivery_lat  48000 non-null  float64
 6   delivery_lon  48000 non-null  float64
 7   distance      48000 non-null  float64
 8   equipment     48000 non-null  object 
 9   weight        47700 non-null  float64
 10  date          48000 non-null  object 
 11  market_index  47626 non-null  float64
 12  quote_signal  48000 non-null  float64
 13  posted_rate   48000 non-null  float64
dtypes: float64(9), object(5)
memory usage: 5.1+ MB
None

--- First 5 Rows ---
     load_id        pickup      delivery  pickup_lat  pickup_lon  \
0  TR-000001      Richmon

In [12]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ---------------------------------------------------------
# 1. Load Data
# Make sure validation.csv, december_chart_inputs.csv, and
# validation_predictions_template.csv are uploaded to Colab!
# ---------------------------------------------------------
train = pd.read_csv("/content/train-test.csv")
val = pd.read_csv("/content/validation.csv")
december = pd.read_csv("/content/december-chart-inputs.csv")

# ---------------------------------------------------------
# 2. Preprocessing & Feature Engineering Function
# ---------------------------------------------------------
def preprocess_and_engineer(df, is_train=True):
    df = df.copy()

    # --- Data Cleaning ---
    # Fix negative weights (convert negative to positive or NaN)
    if 'weight' in df.columns:
        df['weight'] = df['weight'].abs()
        # Impute missing weight with median
        df['weight'] = df['weight'].fillna(df['weight'].median())

    if 'market_index' in df.columns:
        df['market_index'] = df['market_index'].fillna(df['market_index'].median())

    # --- Date Feature Engineering ---
    df['date'] = pd.to_datetime(df['date'])
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_month'] = df['date'].dt.day
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

    # --- Interaction Features ---
    if 'distance' in df.columns and 'weight' in df.columns:
        df['weight_per_mile'] = df['weight'] / (df['distance'] + 1e-5)

    # Categorical encoding for 'equipment', 'pickup', 'delivery'
    for col in ['equipment', 'pickup', 'delivery']:
        if col in df.columns:
            df[col] = df[col].astype('category')

    return df

# Apply preprocessing
train_df = preprocess_and_engineer(train, is_train=True)
val_df = preprocess_and_engineer(val, is_train=False)
december_df = preprocess_and_engineer(december, is_train=False)

# Define feature columns (excluding non-predictive IDs, dates, and target)
features = [
    'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon',
    'distance', 'equipment', 'weight', 'market_index', 'quote_signal',
    'day_of_week', 'day_of_month', 'is_weekend', 'weight_per_mile'
]
target = 'posted_rate'

# Ensure december_df has all feature columns present in train_df for prediction
for col in features:
    if col not in december_df.columns:
        if col in train_df.columns:
            median_val = train_df[col].median()
            december_df[col] = median_val
        else:
            # Fallback for features not in train_df (should ideally not happen)
            december_df[col] = 0.0

X = train_df[features]
y = train_df[target]

# ---------------------------------------------------------
# 3. Model Training with K-Fold Cross Validation
# ---------------------------------------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train_df))
val_preds = np.zeros(len(val_df))
dec_preds = np.zeros(len(december_df))

models = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        num_leaves=31,
        random_state=42,
        verbosity=-1
    )

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )

    oof_preds[val_idx] = model.predict(X_va)
    val_preds += model.predict(val_df[features]) / kf.n_splits
    dec_preds += model.predict(december_df[features]) / kf.n_splits

rmse = np.sqrt(mean_squared_error(y, oof_preds))
mae = mean_absolute_error(y, oof_preds)
print(f"--- Local Validation Performance ---")
print(f"Out-of-Fold RMSE: ${rmse:.2f}")
print(f"Out-of-Fold MAE:  ${mae:.2f}")

# ---------------------------------------------------------
# 4. Format & Save Prediction Files
# ---------------------------------------------------------
# Ensure all predictions are positive (> 0)
val_preds = np.clip(val_preds, 1.0, None)
dec_preds = np.clip(dec_preds, 1.0, None)

# File 1: validation_predictions.csv
val_submission = pd.DataFrame({
    'load_id': val['load_id'],
    'predicted_rate': val_preds
})
val_submission.to_csv("validation_predictions.csv", index=False)
print("\nSaved 'validation_predictions.csv' (Rows: {})".format(len(val_submission)))

# File 2: december_chart_inputs.csv
december['predicted_rate'] = dec_preds
december.to_csv("december_chart_inputs.csv", index=False)
print("Updated 'december_chart_inputs.csv' with predictions.")

--- Local Validation Performance ---
Out-of-Fold RMSE: $601.70
Out-of-Fold MAE:  $138.41

Saved 'validation_predictions.csv' (Rows: 12000)
Updated 'december_chart_inputs.csv' with predictions.


In [11]:
!python score.py --predictions validation_predictions.csv --december-predictions december_chart_inputs.csv

Validated 12,000 final predictions.
Validated 31 fixed December predictions.
Created chart: scorer_results/candidate_december.png
Final validation metrics are calculated by Spotter after submission.
